In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)
library(cowplot)
library(Matrix)
library(Seurat)
library(ggplot2)
library(dplyr)
library(stringr)
library(msigdbr)
library(tidyverse)
library(ggpubr)
library(rstatix)
library(ggprism)
library(reshape2)
library(Seurat)
library(ggplot2)
library(cowplot)
library(tidyverse)
library(dplyr)
library(Seurat)
library(ggplot2)
library(patchwork)
library(ggsci)
library(circlize)
library(RColorBrewer)
library(ComplexHeatmap)
library(scCustomize)
library(ggplot2)
library(ggforce)
library(dplyr)
library(ggrepel)
library(patchwork)
library(ggpubr)
library(export)
library(org.Mm.eg.db)#
library("clusterProfiler")
library(biomaRt)
library(tidyverse)
library(limma)
library(IOBR)
library(fgsea)
library(Seurat)
library(Seurat)
library(ggplot2)
library(dplyr)
library(stringr)
library(DESeq2)
library("FactoMineR")
library(ggplot2)
library(vegan)
library(ape)
library(phyloseq)
library(ggalt)
library(microeco)
#library(jsd)
library("factoextra")
library(Seurat)
library(stringr)
library(ggplot2)
library(pheatmap)
library(dplyr)
library("spatstat.geom")
library(Seurat)
library(Rcpp)
library(harmony)
library(stringr)
library(dplyr)
library(patchwork)
library(ggplot2)
library(ggpubr)
library(reshape2)
library(scales)
library(ggsignif)
library(corrplot)
library(edgeR)
library(ggplot2)
library(FactoMineR)
library(factoextra)  
library(corrplot)   
library(pheatmap)
library(tidyverse)
library(clusterProfiler)
library(org.Hs.eg.db)  
library(GSEABase)    
library(dplyr)
library(Seurat)
library(patchwork)
library(ggplot2)
library(cowplot)
library(monocle)
library(tidyverse)
library(reshape2)
library(ggplot2)
library(ggpubr)
library(GO.db)
library(org.Hs.eg.db)
library(clusterProfiler)
library(enrichplot)
library(ggplot2)
library(ggnewscale)
library(org.Mm.eg.db)
library(dplyr)
library(stringr)
library(ComplexUpset)
options(warn=-1)#

In [ ]:
setwd("./04fig3/")

In [ ]:
sp <- readRDS("./merge_red.RDS")

In [ ]:
sp@meta.data$type <- "other"
sp@meta.data[which(sp@meta.data$celltype=="RGC"),]$type <- "RGC"

In [ ]:
n=0
for(i in unique(sp@meta.data$chip)){
    n=n+1
    sp1 <- subset(sp@meta.data,subset=chip==i)
    cells <- sp1
    all_coords <- cells %>% select(x, y)
    microglia_coords <- cells %>% filter(type == "RGC") %>% select(x, y)
    knn_result <- get.knnx(data = all_coords,
                       query = microglia_coords,
                       k = 11)
    nearest_indices <- knn_result$nn.index[, -1]

    great_ids <- unique(as.vector(nearest_indices))

    cells$type[great_ids] <- "great"
    if(n==1){
        m1 <- cells
    }
    else{
        m1 <-rbind(m1,cells)
    }
}

In [ ]:
sp@meta.data$type <- "NA"
for(i in rownames(m1)){
    type <- m1[which(rownames(m1)==i),]$type
    sp@meta.data[which(rownames(sp@meta.data)==i),]$type <- type
}

In [ ]:
table(sp@meta.data$celltype)

In [ ]:
sam <- subset(sp,subset =type=="great"&area=="GCL")
table(sam@meta.data$celltype)
unique((sam@meta.data$celltype))

In [ ]:
sam1 <- subset(sam,subset = celltype%in%c('RGC','AC','Macroglia','VEC','Microglia','Neutrophil','Myeloid','DC','HC','CBC','RBC','Cone','Rod'))
unique(sam1@meta.data$celltype)
n=0
for(i in unique(sam1@meta.data$celltype)){
    n=n+1
    sam2<-subset(sam1,subset = celltype==i)
    m <- as.data.frame(table(sam2@meta.data$group))
    colnames(m) <- c("sample","Num")
    m$celltype <- i
    if(n==1){
        m1 <- m
    }
    else{
        m1<-rbind(m1,m)
    }   
}

In [ ]:
cols <- c("Sham"="#2b6a99","RIR"="#f16c23")

In [ ]:
order <- c('RGC','AC','Macroglia','VEC','Microglia','Neutrophil','Myeloid','DC','HC','CBC','RBC','Cone',"Rod")
m1$celltype <- factor(m1$celltype,levels = rev(order))

In [ ]:
###plot STAT plot
p1 <- ggplot(m1, mapping = aes(y=Num,x=celltype,fill=sample))+geom_bar(stat='identity',width = 0.6)+
labs(x = "",y = '',title = "") +theme(axis.title =element_text(size = 16),axis.text =element_text(size = 14, color = 'black'))+
theme(
    plot.title= element_text(color = 'black', size   = 20, hjust = 0.5),
    plot.subtitle = element_text(color = 'black', size   = 16,hjust = 0.5),
    plot.caption  = element_text(color = 'black', size   = 16,face = 'italic', hjust = 1),
    axis.text.x   = element_text(color = 'black', size = 16, angle = 45,hjust = 1),
    axis.text.y   = element_text(color = 'black', size = 16, angle = 0),
    axis.title.x  = element_text(color = 'black', size = 16, angle = 0),
    axis.title.y  = element_text(color = 'black', size = 16, angle = 90),
    legend.title  = element_text(color = 'black', size  = 16),
    legend.text   = element_text(color = 'black', size   = 16),
    axis.line.y = element_line(color = 'black', linetype = 'solid'),
    axis.line.x = element_line (color = 'black',linetype = 'solid'), 
    panel.background=element_rect(fill="white"))+scale_fill_manual(values =cols)+
coord_flip()+theme(panel.border = element_rect(color = "black", size = 1, fill = NA))
p1
p2 <- p1+theme(axis.title.y=element_blank(),
        axis.text.y=element_blank(),
        axis.ticks.y=element_blank())
p2

In [ ]:
sam <- subset(sp,subset =type=="great"&area=="GCL")
table(sam@meta.data$celltype)
unique((sam@meta.data$celltype))

In [ ]:
Idents(sam1) <- as.factor(sam1@meta.data$celltype)
deg <- FindAllMarkers(sam1)

In [ ]:
deg[which(deg$cluster=="Macroglia"&deg$avg_log2FC>2&deg$p_val_adj<0.05),]

In [ ]:
gene <- c("Pde6b",###Rod
          "Arr3","Pde6h",##Cone
          "Platr6","Slc7a7",###RBC
          "Krt9","Nfe2",###CBC
          "Ghsr","Galr1","Calb1",##HC
                    "Cd74","H2-DMb1","Ccr7",##DC
                    "Lyz2","Hp","Itgam",##Myeloid
          "S100a8","S100a9",##Neutrophil
          "C1qa","C1qc","Trem2",##Microglia
          "Cldn5","Vwf","Ptprb",###VEC
          "Slc1a3","Cp",##Macroglia
          "Gad1","Slc32a1",##AC
          "Nefl","Sncg","Nefm"##RGC
         )

In [ ]:
sam1@meta.data$celltype <- factor(sam1@meta.data$celltype,levels = rev(order))
Idents(sam1) <- as.factor(sam1@meta.data$celltype)
p <- DotPlot(object = sam1,scale.min=0,
        features=gene)+theme(axis.text.x = element_text(face="italic", hjust=1,angle = 90), axis.text.y = element_text(face="bold")) + 
  scale_colour_gradientn(colours = colorRampPalette(c("#3288BD","white","#D53E4F"))(100))+ theme(legend.position="right")  + 
labs(title = "", y = "", x="")+theme(panel.border = element_rect(color = "black", size = 1, fill = NA))
p

In [ ]:
###plot STAT plot
p1 <- ggplot(m1, mapping = aes(y=Num,x=celltype,fill=sample))+geom_bar(stat='identity',width = 0.6)+
labs(x = "",y = '',title = "") +theme(axis.title =element_text(size = 16),axis.text =element_text(size = 14, color = 'black'))+
theme(
    plot.title= element_text(color = 'black', size   = 20, hjust = 0.5),
    plot.subtitle = element_text(color = 'black', size   = 16,hjust = 0.5),
    plot.caption  = element_text(color = 'black', size   = 16,face = 'italic', hjust = 1),
    axis.text.x   = element_text(color = 'black', size = 16, angle = 45,hjust = 1),
    axis.text.y   = element_text(color = 'black', size = 16, angle = 0),
    axis.title.x  = element_text(color = 'black', size = 16, angle = 0),
    axis.title.y  = element_text(color = 'black', size = 16, angle = 90),
    legend.title  = element_text(color = 'black', size  = 16),
    legend.text   = element_text(color = 'black', size   = 16),
    axis.line.y = element_line(color = 'black', linetype = 'solid'),
    axis.line.x = element_line (color = 'black',linetype = 'solid'), 
    panel.background=element_rect(fill="white"))+scale_fill_manual(values =cols)+
coord_flip()+theme(panel.border = element_rect(color = "black", size = 1, fill = NA))
p1
p2 <- p1+theme(axis.title.y=element_blank(),
        axis.text.y=element_blank(),
        axis.ticks.y=element_blank())
p2
p1

In [ ]:
library(aplot)

In [ ]:
p3 <- p%>%insert_right(p2,width = 0.35)
p3
ggsave("GCL_RGC_near_celltype.pdf",p3,width=12,height = 5.5)